# Air Quality Insight
## Round 2: EDA and Feature Engineering

## 1. Load the Datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

air = pd.read_csv("air_quality.csv")
stations = pd.read_csv("stations.csv")
weather = pd.read_csv("weather.csv")
traffic = pd.read_csv("traffic.csv")
calendar = pd.read_csv("calendar.csv")

print("All datasets loaded successfully!")

In [ ]:
print("Air Quality Dataset")
print(air.head())
print("\nShape:", air.shape)
print("\nData Types:")
print(air.dtypes)
print("\nMissing Values:")
print(air.isnull().sum())
print("\nDuplicate Rows:", air.duplicated().sum())

In [ ]:
print("Summary statistics:")
print(air.describe())

## 2. Data Cleaning

In [ ]:
# Convert timestamp columns to date and time
air["timestamp"] = pd.to_datetime(air["timestamp"])
weather["timestamp"] = pd.to_datetime(weather["timestamp"])
traffic["timestamp"] = pd.to_datetime(traffic["timestamp"])
calendar["timestamp"] = pd.to_datetime(calendar["timestamp"])

# Correct station names
air["station"] = air["station"].str.strip().str.lower()
stations["station"] = stations["station"].str.strip().str.lower()
traffic["station"] = traffic["station"].str.strip().str.lower()

air["station"] = air["station"].replace({
    "central traffic": "central_traffic",
    "industrial east": "industrial_east",
    "market square": "market_square"
})

# Remove duplicate observations
air = air.drop_duplicates(subset=["timestamp", "station"])

print("Data cleaning completed!")

In [ ]:
# Fill missing pollution values with their averages
pollution_columns = ["PM2.5", "PM10", "NO2", "AQI"]

for col in pollution_columns:
    air[col] = air[col].fillna(air[col].mean())

# Fill missing weather values with their averages
weather_columns = ["wind", "temperature", "humidity", "rain"]

for col in weather_columns:
    weather[col] = weather[col].fillna(weather[col].mean())

print("Missing values handled!")
print(air.isnull().sum())

## 3. Combining the Datasets

In [ ]:
# Combine air quality with station information
data = pd.merge(air, stations, on="station", how="left")

# Add weather information
data = pd.merge(data, weather, on="timestamp", how="left")

# Add traffic information
data = pd.merge(
    data, traffic,
    on=["timestamp", "station"],
    how="left"
)

# Add calendar information
data = pd.merge(data, calendar, on="timestamp", how="left")

print("Datasets merged successfully!")
print("Final shape:", data.shape)
print(data.head())

## 4. Exploratory Data Analysis (EDA)

### 4.1 Average AQI at Different Stations

In [ ]:
station_aqi = data.groupby("station")["AQI"].mean()
station_aqi.plot(kind="bar")
plt.title("Average AQI at Different Stations")
plt.xlabel("Station")
plt.ylabel("Average AQI")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4.2 Pollution by Hour

In [ ]:
data["hour"] = data["timestamp"].dt.hour
hour_aqi = data.groupby("hour")["AQI"].mean()
hour_aqi.plot(kind="line", marker="o")
plt.title("Average AQI by Hour")
plt.xlabel("Hour of the Day")
plt.ylabel("Average AQI")
plt.grid()
plt.tight_layout()
plt.show()

### 4.3 Wind Speed and PM2.5

In [ ]:
plt.scatter(data["wind"], data["PM2.5"])
plt.title("Wind Speed vs PM2.5")
plt.xlabel("Wind Speed")
plt.ylabel("PM2.5")
plt.tight_layout()
plt.show()

### 4.4 Traffic Volume and AQI

In [ ]:
plt.scatter(data["traffic_volume"], data["AQI"])
plt.title("Traffic Volume vs AQI")
plt.xlabel("Traffic Volume")
plt.ylabel("AQI")
plt.tight_layout()
plt.show()

### 4.5 Rainfall and AQI

In [ ]:
rain_aqi = data.groupby(
    pd.cut(data["rain"], bins=5, duplicates="drop"),
    observed=False
)["AQI"].mean()
rain_aqi.plot(kind="bar")
plt.title("Rainfall and Average AQI")
plt.xlabel("Rainfall Range")
plt.ylabel("Average AQI")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4.6 Seasonal Pollution Pattern

The supplied synthetic data covers January and February only, so it mainly represents winter. A meaningful comparison across all seasons requires a dataset that includes multiple seasons.

## 5. Correlation Analysis

In [ ]:
columns = ["PM2.5", "PM10", "NO2", "AQI", "wind", "rain", "traffic_volume"]
correlation = data[columns].corr()

print(correlation)

sns.heatmap(correlation, annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 6. Feature Engineering

In [ ]:
# Create simple time-based features
data["day"] = data["timestamp"].dt.day
data["month"] = data["timestamp"].dt.month
data["weekend"] = (data["timestamp"].dt.dayofweek >= 5).astype(int)
data["rush_hour"] = data["hour"].isin([7, 8, 9, 17, 18, 19]).astype(int)

# Convert land-use categories into numeric columns
data = pd.get_dummies(data, columns=["land_use_type"], dtype=int)

print("New features created!")
print(data[["hour", "day", "month", "weekend", "rush_hour"]].head())

## 7. Basic Feature Selection

In [ ]:
# Find which numeric features are related to AQI
numeric_data = data.select_dtypes(include="number")
correlation = numeric_data.corr()["AQI"].abs().sort_values(ascending=False)

print("Features ranked by correlation with AQI:")
print(correlation)

selected_features = correlation[correlation > 0.1].index.tolist()
print("\nSelected features:")
print(selected_features)

## 8. Save the Final Dataset

In [ ]:
data.to_csv("final_air_quality.csv", index=False)
print("Final dataset saved as final_air_quality.csv")
print("Final shape:", data.shape)